# Preflight de entrenamiento desde GCS

Notebook estrictamente read-only para verificar que el dataset migrado a Google Cloud Storage esta listo para alimentar entrenamiento. No sube, borra, sobrescribe ni modifica objetos en GCS.

## Dependencias y configuracion

Verifica dependencias minimas (`google-cloud-storage`, `google-crc32c`, `pandas`, `numpy`, `pillow` y `nibabel` solo cuando el formato SPIDER lo requiera). Usa `PROJECT_ID` y `BUCKET_NAME` explicitos.

In [2]:
from __future__ import annotations
import base64
import hashlib
import importlib.util
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path
from typing import Any

REQUIRED_MODULES = {
    "google.cloud.storage": "google-cloud-storage",
    "google_crc32c": "google-crc32c",
    "pandas": "pandas",
    "numpy": "numpy",
    "PIL": "pillow",
    "SimpleITK": "SimpleITK",
}
missing = [pkg for module, pkg in REQUIRED_MODULES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *sorted(set(missing)),
    ])

import google.auth
import google_crc32c
import numpy as np
import pandas as pd
import SimpleITK as sitk
from google.cloud import storage
from PIL import Image

PROJECT_ID = "pfi-asplanatti-fabrello-v1"
BUCKET_NAME = "pfi-rm-lumbar-artifacts-2026-ef"
BUCKET_URI_PREFIX = f"gs://{BUCKET_NAME}/"
GCS_PREFIX = "datasets/"
EXPECTED_MANIFEST_SHA256 = "fa54c89a278d850021c0f91c0a27b3b5211c86301c9e4f125d75d517f39b793b"
EXPECTED_MANIFEST_ROWS = 2058
EXPECTED_RECEIPT_ROWS = 2058
EXPECTED_BUCKET_DATASET_OBJECTS = 2059
EXPECTED_COMPONENT_COUNTS = {"spider_image": 447, "spider_mask": 447, "axial_image": 610, "axial_mask": 552}
EXPECTED_MANIFEST_COLUMNS = ["component", "source_path", "source_root", "relative_path", "destination_uri", "suffix", "size_bytes", "modified_at", "sha256", "referenced_rows", "exists", "is_symlink", "status"]
RECEIPT_COLUMNS = ["manifest_sha256", "component", "source_path", "relative_path", "destination_uri", "object_name", "size_bytes", "crc32c", "md5_hash", "generation", "updated", "uploaded_at_utc", "upload_status"]
DRIVE_ROOT = Path("/content/drive")
MY_DRIVE = DRIVE_ROOT / "MyDrive"
PFI_ROOT = MY_DRIVE / "PFI_MVP"
DEFAULT_PLAN_DIR = PFI_ROOT / "results" / "GCS_dataset_migration_plan"
DEFAULT_UPLOAD_DIR = PFI_ROOT / "results" / "GCS_dataset_upload"
PLAN_DIR = Path(os.getenv("PFI_GCS_PLAN_DIR", str(DEFAULT_PLAN_DIR)))
UPLOAD_STATE_DIR = Path(os.getenv("PFI_GCS_UPLOAD_STATE_DIR", str(DEFAULT_UPLOAD_DIR)))
MANIFEST_CSV = Path(os.getenv("PFI_GCS_MANIFEST_CSV", str(PLAN_DIR / "gcs_upload_manifest.csv")))
RECEIPT_CSV = Path(os.getenv("PFI_GCS_RECEIPT_CSV", str(UPLOAD_STATE_DIR / "gcs_upload_receipt.csv")))
STATE_JSON = Path(os.getenv("PFI_GCS_UPLOAD_STATE_JSON", str(UPLOAD_STATE_DIR / "gcs_upload_state.json")))
FINAL_SUMMARY_JSON = Path(os.getenv("PFI_GCS_FINAL_SUMMARY_JSON", str(UPLOAD_STATE_DIR / "gcs_upload_final_summary.json")))
SPIDER_TRAINING_INDEX_CSV = Path("/content/spider_training_index.csv")
SAMPLE_DIR = Path(tempfile.mkdtemp(prefix="pfi_spider_preflight_"))
print(json.dumps({"PROJECT_ID": PROJECT_ID, "BUCKET_NAME": BUCKET_NAME, "manifest": str(MANIFEST_CSV), "receipt": str(RECEIPT_CSV), "sample_dir": str(SAMPLE_DIR)}, indent=2))

{
  "PROJECT_ID": "pfi-asplanatti-fabrello-v1",
  "BUCKET_NAME": "pfi-rm-lumbar-artifacts-2026-ef",
  "manifest": "/content/drive/MyDrive/PFI_MVP/results/GCS_dataset_migration_plan/gcs_upload_manifest.csv",
  "receipt": "/content/drive/MyDrive/PFI_MVP/results/GCS_dataset_upload/gcs_upload_receipt.csv",
  "sample_dir": "/tmp/pfi_spider_preflight_mtul6vtg"
}


## Manifest, receipt y estado local

Carga desde Google Drive o rutas configurables el manifiesto original, `gcs_upload_receipt.csv`, `gcs_upload_state.json` y `gcs_upload_final_summary.json` si existe. Valida SHA, filas, duplicados y conteos congelados.

## Autenticacion y bucket read-only

En Colab usa `google.colab.auth.authenticate_user()`. En VM usa Application Default Credentials. No usa claves JSON ni service account keys descargadas. La inspeccion del bucket es solo lectura.

## Verificacion remota e indice SPIDER

Lista `datasets/`, exige 2059 objetos incluyendo el marcador vacio `datasets/`, verifica tama?o y CRC32C de los 2058 objetos del receipt, y genera localmente `/content/spider_training_index.csv`.

## Muestras deterministicas

Descarga temporalmente el primer par, el par medio y el ultimo par SPIDER; abre imagen y mascara, e imprime dimensiones, dtype y valores unicos de mascara sin rutas largas ni informacion identificatoria.

In [3]:
def ensure_drive_available() -> None:
    if MY_DRIVE.exists():
        return
    try:
        from google.colab import drive
    except ImportError:
        return
    drive.mount(str(DRIVE_ROOT))

def require_file(path: Path) -> None:
    if not path.exists() or not path.is_file():
        raise FileNotFoundError(path)

def sha256_stream(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def destination_object_name(uri: str) -> str:
    if not isinstance(uri, str) or not uri.startswith(BUCKET_URI_PREFIX):
        raise ValueError(f"destination_uri fuera del bucket esperado: {uri}")
    name = uri[len(BUCKET_URI_PREFIX):]
    if not name.startswith(GCS_PREFIX):
        raise ValueError(f"Objeto fuera de {GCS_PREFIX}: {uri}")
    return name

def normalized_pair_key(path_like: str) -> str:
    stem = Path(path_like).stem.lower()
    for suffix in ("_image", "_mask", "_label"):
        if stem.endswith(suffix):
            stem = stem[:-len(suffix)]
    return stem

def validate_manifest() -> tuple[pd.DataFrame, str]:
    require_file(MANIFEST_CSV)
    digest = sha256_stream(MANIFEST_CSV)
    if digest != EXPECTED_MANIFEST_SHA256:
        raise ValueError(f"SHA de manifest inesperado: {digest}")
    manifest = pd.read_csv(MANIFEST_CSV, keep_default_na=False)
    if list(manifest.columns) != EXPECTED_MANIFEST_COLUMNS:
        raise ValueError("Columnas inesperadas en manifest")
    if len(manifest) != EXPECTED_MANIFEST_ROWS:
        raise ValueError(f"Filas de manifest inesperadas: {len(manifest)}")
    manifest["size_bytes"] = pd.to_numeric(manifest["size_bytes"], errors="raise").astype("int64")
    if manifest["destination_uri"].duplicated().any():
        raise ValueError("destination_uri duplicados en manifest")
    if manifest.duplicated(["component", "relative_path"]).any():
        raise ValueError("relative_path duplicado dentro de componente")
    counts = manifest["component"].value_counts().to_dict()
    for component, expected in EXPECTED_COMPONENT_COUNTS.items():
        if int(counts.get(component, 0)) != expected:
            raise ValueError(f"Conteo inesperado para {component}: {counts.get(component, 0)}")
    manifest["object_name"] = manifest["destination_uri"].map(destination_object_name)
    return manifest, digest

def validate_receipt(manifest: pd.DataFrame) -> pd.DataFrame:
    require_file(RECEIPT_CSV)
    receipt = pd.read_csv(RECEIPT_CSV, keep_default_na=False)
    if list(receipt.columns) != RECEIPT_COLUMNS:
        raise ValueError("Columnas inesperadas en receipt")
    if len(receipt) != EXPECTED_RECEIPT_ROWS:
        raise ValueError(f"Filas de receipt inesperadas: {len(receipt)}")
    if receipt["destination_uri"].duplicated().any():
        raise ValueError("destination_uri duplicados en receipt")
    if not bool((receipt["manifest_sha256"] == EXPECTED_MANIFEST_SHA256).all()):
        raise ValueError("Receipt contiene SHA de manifest distinto")
    manifest_by_dest = manifest.set_index("destination_uri")
    for row in receipt.itertuples(index=False):
        if row.destination_uri not in manifest_by_dest.index:
            raise ValueError(f"Receipt sin manifest: {row.destination_uri}")
        src = manifest_by_dest.loc[row.destination_uri]
        if str(row.component) != str(src["component"]) or str(row.relative_path) != str(src["relative_path"]) or str(row.object_name) != str(src["object_name"]) or int(row.size_bytes) != int(src["size_bytes"]):
            raise ValueError(f"Receipt no coincide con manifest: {row.destination_uri}")
    return receipt

def load_optional_json(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    require_file(path)
    with path.open("r", encoding="utf-8") as fh:
        return json.load(fh)

def authenticate_storage_client() -> storage.Client:
    try:
        from google.colab import auth
        auth.authenticate_user()
    except ImportError:
        pass
    credentials, auth_project = google.auth.default()
    print(json.dumps({"auth_project_detected": auth_project, "client_project": PROJECT_ID}, indent=2))
    return storage.Client(project=PROJECT_ID, credentials=credentials)

def list_dataset_objects(client: storage.Client) -> tuple[dict[str, Any], dict[str, dict[str, Any]]]:
    bucket = client.bucket(BUCKET_NAME)
    bucket.reload(client=client)
    metadata = {"name": bucket.name, "location": str(bucket.location or ""), "storage_class": str(bucket.storage_class or "")}
    objects = {}
    for blob in client.list_blobs(BUCKET_NAME, prefix=GCS_PREFIX):
        objects[blob.name] = {"name": blob.name, "size": int(blob.size or 0), "crc32c": blob.crc32c, "generation": str(blob.generation) if blob.generation is not None else None, "updated": blob.updated.isoformat() if blob.updated is not None else None}
    return metadata, objects

def verify_bucket_objects(manifest: pd.DataFrame, receipt: pd.DataFrame, objects: dict[str, dict[str, Any]]) -> tuple[list[str], list[str], int]:
    planned = set(manifest["object_name"])
    placeholder = objects.get(GCS_PREFIX)
    placeholder_ok = placeholder is not None and int(placeholder["size"]) == 0
    unexpected = sorted(set(objects) - planned - {GCS_PREFIX})
    missing = sorted(planned - set(objects))
    if len(objects) != EXPECTED_BUCKET_DATASET_OBJECTS or not placeholder_ok:
        raise ValueError("Cantidad de objetos o marcador datasets/ inesperado")
    receipt_by_object = receipt.set_index("object_name")
    verified = 0
    total = len(receipt_by_object)
    for index, (object_name, row) in enumerate(receipt_by_object.iterrows(), start=1):
        remote = objects.get(object_name)
        if remote is None:
            raise ValueError(f"Objeto faltante: {object_name}")
        if int(remote["size"]) != int(row["size_bytes"]) or str(remote["crc32c"]) != str(row["crc32c"]):
            raise ValueError(f"Metadata remota no coincide con receipt: {object_name}")
        verified += 1
        if index % 250 == 0 or index == total:
            print(f"Objetos verificados contra receipt: {index}/{total}")
    return unexpected, missing, verified

def build_spider_training_index(manifest: pd.DataFrame, receipt: pd.DataFrame) -> pd.DataFrame:
    spider = manifest[manifest["component"].isin(["spider_image", "spider_mask"])].copy()
    spider["pair_id"] = spider["relative_path"].map(normalized_pair_key)
    joined = spider.merge(receipt[["destination_uri", "crc32c", "size_bytes"]], on="destination_uri", how="left", suffixes=("", "_receipt"), validate="one_to_one")
    rows = []
    for pair_id, group in joined.groupby("pair_id", sort=True):
        if set(group["component"]) != {"spider_image", "spider_mask"} or len(group) != 2:
            raise ValueError(f"Par SPIDER incompleto: {pair_id}")
        image = group[group["component"] == "spider_image"].iloc[0]
        mask = group[group["component"] == "spider_mask"].iloc[0]
        rows.append({"pair_id": pair_id, "image_gcs_uri": image["destination_uri"], "mask_gcs_uri": mask["destination_uri"], "image_size_bytes": int(image["size_bytes"]), "mask_size_bytes": int(mask["size_bytes"]), "image_crc32c": image["crc32c"], "mask_crc32c": mask["crc32c"]})
    index_df = pd.DataFrame(rows).sort_values("pair_id", kind="stable").reset_index(drop=True)
    if len(index_df) != EXPECTED_COMPONENT_COUNTS["spider_image"]:
        raise ValueError("Cantidad de pares SPIDER inesperada")
    index_df.to_csv(SPIDER_TRAINING_INDEX_CSV, index=False)
    return index_df

def download_uri(client: storage.Client, uri: str, target: Path) -> Path:
    object_name = destination_object_name(uri)
    client.bucket(BUCKET_NAME).blob(object_name).download_to_filename(str(target))
    return target

def open_medical_array(path: Path) -> np.ndarray:
    suffixes = [suffix.lower() for suffix in path.suffixes]
    suffix = path.suffix.lower()
    if suffix == ".npy":
        return np.load(path)
    if suffix in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}:
        return np.asarray(Image.open(path))
    if suffix in {".mha", ".mhd"}:
        image = sitk.ReadImage(str(path))
        return sitk.GetArrayFromImage(image)
    if suffix == ".nii" or suffixes[-2:] == [".nii", ".gz"]:
        if importlib.util.find_spec("nibabel") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "nibabel"])
        import nibabel as nib
        return np.asarray(nib.load(str(path)).get_fdata())
    raise RuntimeError(f"Formato de muestra no soportado en preflight: {suffix}")

def sample_spider_pairs(client: storage.Client, index_df: pd.DataFrame) -> int:
    sample_positions = [0, len(index_df) // 2, len(index_df) - 1]
    opened = 0
    for pos in sample_positions:
        row = index_df.iloc[pos]
        safe_pair = f"sample_{pos}"
        image_path = download_uri(client, row["image_gcs_uri"], SAMPLE_DIR / f"{safe_pair}_image{Path(destination_object_name(row['image_gcs_uri'])).suffix}")
        mask_path = download_uri(client, row["mask_gcs_uri"], SAMPLE_DIR / f"{safe_pair}_mask{Path(destination_object_name(row['mask_gcs_uri'])).suffix}")
        image_arr = open_medical_array(image_path)
        mask_arr = open_medical_array(mask_path)
        if image_arr.size <= 0 or mask_arr.size <= 0:
            raise ValueError("Muestra SPIDER vacia")
        if image_arr.shape != mask_arr.shape:
            raise ValueError(f"Shape mismatch en muestra SPIDER: image={image_arr.shape} mask={mask_arr.shape}")
        if image_arr.ndim not in {2, 3} or mask_arr.ndim not in {2, 3}:
            raise ValueError(f"Dimensiones invalidas en muestra SPIDER: image={image_arr.ndim} mask={mask_arr.ndim}")
        unique_mask_values = np.unique(mask_arr)
        if len(unique_mask_values) == 0:
            raise ValueError("Mascara SPIDER sin valores unicos")
        print(json.dumps({"sample_position": int(pos), "image_shape": list(image_arr.shape), "image_dtype": str(image_arr.dtype), "mask_shape": list(mask_arr.shape), "mask_dtype": str(mask_arr.dtype), "mask_unique_values": [int(v) if float(v).is_integer() else float(v) for v in unique_mask_values[:30]], "mask_unique_count": int(len(unique_mask_values))}, indent=2))
        opened += 1
    return opened

ensure_drive_available()
manifest_df, manifest_sha256 = validate_manifest()
receipt_df = validate_receipt(manifest_df)
state_payload = load_optional_json(STATE_JSON)
final_summary_payload = load_optional_json(FINAL_SUMMARY_JSON)
client = authenticate_storage_client()
bucket_metadata, dataset_objects = list_dataset_objects(client)
unexpected_objects, missing_objects, expected_objects_verified = verify_bucket_objects(manifest_df, receipt_df, dataset_objects)
spider_index_df = build_spider_training_index(manifest_df, receipt_df)
sampled_spider_pairs_opened = sample_spider_pairs(client, spider_index_df)
counts = manifest_df["component"].value_counts().to_dict()
final_json = {"manifest_sha256": manifest_sha256, "manifest_rows": int(len(manifest_df)), "receipt_rows": int(len(receipt_df)), "bucket_dataset_objects": int(len(dataset_objects)), "expected_objects_verified": int(expected_objects_verified), "unexpected_objects": len(unexpected_objects), "missing_objects": len(missing_objects), "spider_images": int(counts.get("spider_image", 0)), "spider_masks": int(counts.get("spider_mask", 0)), "spider_complete_pairs": int(len(spider_index_df)), "axial_images": int(counts.get("axial_image", 0)), "axial_masks": int(counts.get("axial_mask", 0)), "sampled_spider_pairs_opened": int(sampled_spider_pairs_opened), "TRAINING_PREFLIGHT_SUCCESS": bool(manifest_sha256 == EXPECTED_MANIFEST_SHA256 and len(manifest_df) == EXPECTED_MANIFEST_ROWS and len(receipt_df) == EXPECTED_RECEIPT_ROWS and expected_objects_verified == EXPECTED_RECEIPT_ROWS and not unexpected_objects and not missing_objects and len(spider_index_df) == EXPECTED_COMPONENT_COUNTS["spider_image"] and sampled_spider_pairs_opened == 3)}
print(json.dumps(final_json, indent=2))

Mounted at /content/drive
{
  "auth_project_detected": "",
  "client_project": "pfi-asplanatti-fabrello-v1"
}
Objetos verificados contra receipt: 250/2058
Objetos verificados contra receipt: 500/2058
Objetos verificados contra receipt: 750/2058
Objetos verificados contra receipt: 1000/2058
Objetos verificados contra receipt: 1250/2058
Objetos verificados contra receipt: 1500/2058
Objetos verificados contra receipt: 1750/2058
Objetos verificados contra receipt: 2000/2058
Objetos verificados contra receipt: 2058/2058
{
  "sample_position": 0,
  "image_shape": [
    797,
    492,
    21
  ],
  "image_dtype": "int16",
  "mask_shape": [
    797,
    492,
    21
  ],
  "mask_dtype": "int16",
  "mask_unique_values": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    100,
    201,
    202,
    203,
    204,
    205,
    206,
    207,
    208
  ],
  "mask_unique_count": 18
}
{
  "sample_position": 223,
  "image_shape": [
    21,
    512,
    512
  ],
  "image_dtype": "uint16",

## Resultado

`TRAINING_PREFLIGHT_SUCCESS` solo puede ser `true` si coinciden SHA, manifest, receipt, objetos esperados, pares SPIDER completos y las 3 muestras se abren correctamente.